In [ ]:
from datetime import date, timedelta
import json
import os
from pathlib import Path
import time

from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
import exactextract
import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pycountry
import requests
from shapely.geometry import mapping

In [ ]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_out_folder = base_dir + "data_out/"

# out folder for maps (hazards)
maps_hazards_folder = base_dir + "data_out/maps/hazards/"

file_countries_acled = "acled_iso_codes.xlsx"

# shapes subfolder
data_in_shapes = base_dir + "data_in/shapes/"

# out folder for maps (population)
maps_pop_folder = base_dir + "data_out/maps/PIN_population/"


In [ ]:
# read acled country names
df_acled_countries = pd.read_excel(
    data_in_folder + file_countries_acled,
    dtype=str,
)

# dictionary for acled country names not in pycountry
pycountry_acled_map = {
    "Democratic Republic of Congo": "Congo, The Democratic Republic of the",
    "Republic of Congo": "Congo",
    "Saint Vincent and Grenadines": "Saint Vincent and the Grenadines",
    "Caribbean Netherlands": "Bonaire, Sint Eustatius and Saba",
    "Bailiwick of Guernsey": "Guernsey",
    "Bailiwick of Jersey": "Jersey",
    "Laos": "Lao People's Democratic Republic",
    "East Timor": "Timor-Leste",
    "Ivory Coast": "Côte d'Ivoire",
    "Cape Verde": "Cabo Verde",
    "Turkey": "Türkiye",
}

# get ISO3 codes from pycountry (no error handling: assumes all are found)
df_acled_countries['iso3'] = df_acled_countries['Country'].apply(
    # introduced Niger exception (known case to avoid Nigeria)
    lambda country: pycountry.countries.search_fuzzy(
        pycountry_acled_map.get(country, country)
    )[0].alpha_3 if country != "Niger" else pycountry.countries.search_fuzzy(
        # introduced Niger exception (known case to avoid Nigeria)
        pycountry_acled_map.get(country, country)
    )[1].alpha_3
)


In [ ]:
load_dotenv()  # reads .env

# access acled user and token from secrets (modified July 2025)
un_email = os.environ["aps_acled_email"]

# no longer exists (July 2025)
# acled_tkn = dbutils.secrets.get(scope = "aps-acled-api", key = "usertoken")

# password from myAcled re-registration
acled_psw = os.environ["aps_acled_psw"]

# new acled oauth token
acled_token_url = "https://acleddata.com/oauth/token"

# client payload
acled_id_payload = {
    "username": un_email,
    "password": acled_psw,
    "grant_type": "password",
    "client_id": "acled",
}

# http headers
acled_id_headers = {
    'Content-Type': 'application/x-www-form-urlencoded',
}

# get access token with post request
# TODO: error handling
acled_token_resp = requests.post(
    url=acled_token_url,
    headers=acled_id_headers,
    data=acled_id_payload,
)
token_resp = acled_token_resp.json()


In [ ]:
# new acled url endpoint
acled_url_end = "https://acleddata.com/api/"

# get aoi headers
acled_headers = {
    'Content-Type': 'application/json',
    'Authorization': f"{token_resp['token_type']} {token_resp['access_token']}",
}

# acled read
acled_data = "acled/read"

# NOTE: select last year or last 12 months of data
time_filter = "last_year" # "last_12_months" #

# Find the date of the last updated Friday using modulo operation (assumes fresh data on Wednesdays)
today = date.today()
weekday = today.weekday()
last_updated_friday = today - timedelta(
    days=(weekday - 4) % 7 + (0 if weekday in [2, 3] else 7)
)

# end of previous month with completed updates or end of last year
last_updated_month = (
    last_updated_friday - timedelta(days=last_updated_friday.day)
    if time_filter == "last_12_months"
    else date(today.year - 1, 12, 31)
)
# 12 last completed months with updates
start_query_date = last_updated_month - relativedelta(months=12) + timedelta(days=1)

# NOTE: selected countries for WIA execution
wia_countries_exercise = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    "Democratic Republic of Congo",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and Grenadines",
    # "Grenada",
]

# NOTE: select admin level
admin_level = "2"
year = f"{today.year - 1}"

# NOTE no error handling: selected country name is not the acled one
c_iso3 = df_acled_countries.query(
    "Country in @wia_countries_exercise"
)['iso3'].iloc[0]


In [ ]:
# acled query params (filter countries between dates)
acled_query = {
    # country query must be created with concatenated OR's
    # NOTE: apparently modified July 2025 with OR operator (|)
    "country": ["|".join(wia_countries_exercise)],

    # country query default is "LIKE", modify to "=" ("%3D")
    "country_where": ["%3D"], # [], #
    "event_date": [f"{start_query_date}|{last_updated_month}"],
    "event_date_where": ["BETWEEN"],

    # selected event types: Battles, Explosions/Remote violence, Violence against civilians
    # "event_type": ["Battles", "Explosions/Remote violence", "Violence against civilians"],
    # NOTE: apparently modified July 2025 with OR operator (|)
    "event_type": ["|".join(
        ["Battles", "Explosions/Remote violence", "Violence against civilians"]
    )],

    "event_type_where": ["%3D"], # [], #
}

# construct query parameters from dictionary
acled_query_params = "&".join(
    [
        f"{key}=" + f":OR:{key}=".join(map(str, value))
        for key, value in acled_query.items()
        # add parameter if not empty list in value
        if value
    ]
)


In [ ]:
def make_paginated_acled_call(
    url,
    auth_headers=None,
    max_attempts=5,
    sleep_duration=2,
):
    """
    Makes a paginated acled API call and returns results in a DataFrame.

    :param url: URL for the API call (including http parameters).
    :param auth_headers: Authentication headers for the API call (new for acled July 2025).
    :param max_attempts: Maximum number of attempts for the API call.
    :param sleep_duration: Duration (in seconds) to sleep between attempts.
    :return: DataFrame with all results, or None if the call fails.
    """

    all_data = pd.DataFrame()
    page = 1    # init page
    n_rows = 5000   # init number of returned rows by acled request
    default_acled_limit = 5000  # limit of 5000 returned data rows by default
    # grab params from url
    init_params = url.split("?")[1]
    base_url = url.split("?")[0]
    
    # break assumption: last page is reached when n_rows < default limit
    while n_rows == default_acled_limit:
        for attempt in range(max_attempts):

            try:
                response = requests.get(
                    base_url + "?" + init_params + f"&page={page}",
                    headers=auth_headers,
                )
                response.raise_for_status()
            except requests.exceptions.RequestException as e:
                # don't print "attempts failure"
                # print(f"API request failed: {e}")
                response = None
            
            if response and response.status_code == 200:
                data = pd.DataFrame(response.json()["data"])
                all_data = pd.concat([all_data, data], ignore_index=True)
                # n_rows updated
                n_rows = response.json()['count']
                break
            else:
                time.sleep(sleep_duration)

        if response is None:
            print(f"Failed to retrieve {url} after {max_attempts} attempts.")
            return None

        page += 1

    return all_data


In [ ]:
# acled request: countries in list, last updated 12 months
acled_df = make_paginated_acled_call(
    acled_url_end + acled_data + "?" + acled_query_params,
    acled_headers,
)

# Check if DataFrame None or empty
if acled_df is None:
    raise Exception("Stop execution: ACLED API call failed")
elif acled_df.empty:
    raise Exception("Stop execution: no conflict reported for query")


In [ ]:
# Function to determine buffer size based on event type and fatalities
def get_buffer_distance(event_type, fatality):
    # 5 km for Battles and Explosions/Remote violence
    if event_type in ["Battles", "Explosions/Remote violence"]:
        return 5
    # 5 km for Violence against civilians with fatalities
    elif event_type == "Violence against civilians" and fatality > 0:
        return 5
    # 2 km for Violence against civilians without fatalities or Riots
    elif event_type in ["Violence against civilians", "Riots"]:
        return 2
    # 1 km for Protests
    else:
        return 1

# use get_buffer_distance to create buffer column for each event
# TODO: in large data, this could be done in spark
acled_df['buff_col'] = acled_df.apply(
    lambda row: get_buffer_distance(row.event_type, row.fatalities),
    axis=1,
)


In [ ]:
# geojson output file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
# Get geojson in shapes
geojson_file = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read country shape geojson into gdf
gdf = gpd.read_file(
    data_in_shapes + f"{geojson_file[0]}"
).to_crs('EPSG:4326')

# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"


In [ ]:
# DBFS root
dbfs_base_dir = "./"
# WorldPop folder
wpop_dir = "WorldPop_rasters"
# indiv country rasters folder (constrained - total population)
country_pop_dir = "G2_CN_POP_R25A_100m"
# list of total_pop rasters available in DBFS
total_pop_rasters = [
    f.name
    for f in Path(
        data_in_folder + wpop_dir + "/" + country_pop_dir
    ).iterdir()
    if f.is_file()
]

# identify raster id for the country
# NOTE: assumes raster already downloaded in DBFS
country_raster_id = [
    data_in_folder + wpop_dir + "/" + country_pop_dir + "/" + next(
        (
            f for f in total_pop_rasters
            if (c_iso3.lower() in f) and (f"_{year}_" in f)
        ),
        "None"
    )
]


In [ ]:
# drop duplicated to alivate hazard shape
# if repeated events location, keep the most severe buffer
violent_df = acled_df.sort_values(
    by="buff_col"
).drop_duplicates(
    subset=["latitude", "longitude"],
    keep="last",
)

# Overlay - computes intersection geometries
impact_gdf = gpd.overlay(
    gdf,
    gpd.GeoDataFrame(
        geometry=[gpd.GeoSeries(
            gpd.points_from_xy(violent_df.longitude, violent_df.latitude),
            # CRS should be EPSG:4326 as geodata is in lat/lon four decimal degrees
            crs="EPSG:4326",
        ).to_crs(
            # Project EPSG:3857 for buffering in meters and union polygon from result
            "EPSG:3857"
        ).buffer(
            distance=violent_df.buff_col.values*1000
        ).union_all(
            # default option: robust for overlapping but slow for large numbers of geometries
            method="unary"
        )],
        crs="EPSG:3857",
    ).to_crs("EPSG:4326"),
    how='intersection'
)


In [ ]:
# read population data for country/admin/year
# merge with violence extract compute
# keep non-affected zones: assumes no missing pcodes in pop data
violence_impact_df = pd.read_excel(
    maps_pop_folder + f"{c_iso3}_adm{admin_level}_{year}_WorldPop.xlsx",
).astype(
    {pcode_col: str}
).merge(
    exactextract.exact_extract(
        country_raster_id,
        [
            {
                "geometry": mapping(g),
                "properties": {
                    f"{pcode_col}": g_id,
                    f"{name_col}": g_n,
                },
            } for g, g_id, g_n  in zip(
                impact_gdf.geometry,
                impact_gdf[pcode_col],
                impact_gdf[name_col]
            )
        ],
        'sum',
        include_cols=[f"{pcode_col}", f"{name_col}"],
        output="pandas",
    ).rename(
        columns={
            "sum": f"impact_violence_pop_{year}",
        }
    ),
    on=[pcode_col, name_col],
    how="left",
).sort_values(
    pcode_col
).fillna(
    {f"impact_violence_pop_{year}": 0}
)


In [ ]:
# Check if there're none pop tiles for a pcode
# NOTE: exact extract may return null or zero if none pop tiles for a pcode
if len(violence_impact_df) != len(gdf):
    raise Exception("Stop execution: There're no pop tiles for pcode(s)")

In [ ]:
# compute population rate in new column
# NOTE: save division by zero with fillna zeros
violence_impact_df[f"impact_violence_share_{year}"] = (
    violence_impact_df[f"impact_violence_pop_{year}"] / violence_impact_df[f"tot_pop_{year}"] * 100
).round(2).fillna(0)

# 0/1 binary mapping for wia: below or above mean/median
imp_share_mean = violence_impact_df[f"impact_violence_share_{year}"].mean()
imp_share_med = violence_impact_df[f"impact_violence_share_{year}"].median()

# series mapping into new columns
violence_impact_df["share_above_mean"] = (violence_impact_df[f"impact_violence_share_{year}"] >= imp_share_mean).astype(int)
violence_impact_df["share_above_med"] = (violence_impact_df[f"impact_violence_share_{year}"] >= imp_share_med).astype(int)

# NOTE: assumes impact share has no nulls (should be 0 where no impact)

# acled query reference time_period
violence_impact_df["ref_date"] = f"{last_updated_month}"


In [ ]:
# save violence impact data
violence_impact_df[
    [
        pcode_col,
        name_col,
        f"impact_violence_share_{year}",
        "share_above_mean",
        "share_above_med",
        "ref_date",
    ]
].to_excel(
    maps_hazards_folder + f"{c_iso3}_adm{admin_level}_{year}_ACLED.xlsx",
    index=False,
    sheet_name=f"{c_iso3}_adm{admin_level}_{year}",
)


In [ ]:
# Disable automatic rendering
pio.renderers.default = None

# plotly express choropleth map of country population share impact violence by admin area polygon
vio_impact_share_fig = px.choropleth(
    violence_impact_df,
    geojson=json.loads(gdf.to_json()),
    featureidkey=f"properties.{pcode_col}",
    locations=pcode_col,
    color=f"impact_violence_share_{year}",
    color_continuous_scale="YlOrRd",
    projection="mercator",
    width=1200,
    height=900,
)
vio_impact_share_fig.update_geos(fitbounds="locations", visible=False)
vio_impact_share_fig.update_layout(
    title=dict(
        text=f"Impact Violence Pop. Share {year} ({c_iso3}_adm{admin_level})",
        x=0.5,  # x position (0-left, 1-right)
        y=0.97,  # y position (0-bottom, 1-top in normalized coordinates)
    ),
    margin={"r": 0, "t": 0, "l": 0, "b": 0}
)
vio_impact_share_fig.update_coloraxes(colorbar_len=0.65)
vio_impact_share_fig.update_traces(marker_line_color="Gainsboro", marker_line_width=0.5)


In [ ]:
# write figure as png direct locally
vio_impact_share_fig.write_image(
    maps_hazards_folder + f"{c_iso3}_adm{admin_level}_{year}_ACLED.png",
    format="png",
    scale=2,
)
